In [1]:
import os
import hydra
import json
import bpy
#sudo apt-get install ffmpeg
from omegaconf import OmegaConf,open_dict
import omegaconf

In [2]:

config_home_dir = '/users/j/a/jahn25/puzzlepp/config'
config_home_dir = '../config'

cfg_auto_aggl = omegaconf.OmegaConf.load(config_home_dir+'/auto_aggl.yaml')


cfg = OmegaConf.merge( cfg_auto_aggl )

data_home_dir = '/disk2/data/breaking_bad/'
data_home_dir = '/data/jhahn/data/shape_dataset/'

cfg.experiment_output_path = ''
cfg.inference_dir= data_home_dir+'results'
cfg.renderer.output_path = data_home_dir+'results_render/'
cfg.renderer.mesh_path = f'{data_home_dir}/data/'
cfg.ffmpeg_path = '/usr/bin/ffmpeg'

In [25]:
import importlib
import myrenderer
importlib.reload(myrenderer)

from myrenderer import MyRenderer
import numpy as np
from mathutils import Quaternion, Vector, Matrix

In [33]:
_centroid

array([0.65711981, 0.40361999, 0.2609222 ])

In [34]:
Matrix.Translation(Vector([0,1,0]))

Matrix(((1.0, 0.0, 0.0, 0.0),
        (0.0, 1.0, 0.0, 1.0),
        (0.0, 0.0, 1.0, 0.0),
        (0.0, 0.0, 0.0, 1.0)))

In [41]:
print(_centroid)
print(Vector(_centroid) + Matrix.Translation(Vector([0,1,0])).to_translation())

[0.65711981 0.40361999 0.2609222 ]
<Vector (0.6571, 1.4036, 0.2609)>


In [17]:
mesh = parts[0].data

print("# of vertices=%d" % len(mesh.vertices))
p = []
for vert in mesh.vertices:
    print( 'v %f %f %f\n' % (vert.co.x, vert.co.y, vert.co.z) )
    p.append([vert.co.x, vert.co.y, vert.co.z])

print("# of faces=%d" % len(mesh.polygons))

for face in mesh.polygons:
    print('face')
    #dump(face)
    for vert in face.vertices:
        print(vert)

# of vertices=6301
v 0.456885 0.524566 -0.078997

v 0.457096 0.524566 -0.079183

v 0.457165 0.524478 -0.079157

v 0.458659 0.524566 -0.080024

v 0.485049 0.481256 -0.077072

v 0.492399 0.486690 -0.084923

v 0.496175 0.481905 -0.083521

v 0.500258 0.492137 -0.091896

v 0.505799 0.485770 -0.089149

v 0.509132 0.497610 -0.097114

v 0.514673 0.491243 -0.094368

v 0.518006 0.503083 -0.102333

v 0.523548 0.496717 -0.099586

v 0.527896 0.508583 -0.105796

v 0.531671 0.503798 -0.104394

v 0.538293 0.514096 -0.108382

v 0.471671 0.485432 -0.069392

v 0.478006 0.490839 -0.078998

v 0.485610 0.496280 -0.086410

v 0.494061 0.501742 -0.092359

v 0.502935 0.507215 -0.097578

v 0.511809 0.512688 -0.102796

v 0.521107 0.518172 -0.107283

v 0.531250 0.523679 -0.110308

v 0.541319 0.524566 -0.110290

v 0.533078 0.524566 -0.110441

v 0.461251 0.491432 -0.063452

v 0.464628 0.495015 -0.071318

v 0.471809 0.500444 -0.079461

v 0.479583 0.505889 -0.086581

v 0.488249 0.511357 -0.092158

v 0.497124 0.516830 

In [13]:


from pprint import pprint
#dir(parts[0])

for attr in dir(parts[0]):
    print("obj.%s = %r" % (attr, getattr(parts[0], attr)))

obj.__doc__ = None
obj.__module__ = 'bpy_types'
obj.__slots__ = ()
obj.active_material = bpy.data.materials['MeshMaterial']
obj.active_material_index = 0
obj.active_shape_key = None
obj.active_shape_key_index = 0
obj.add_rest_position_attribute = False
obj.animation_data = None
obj.animation_data_clear = <bpy_func Object.animation_data_clear()>
obj.animation_data_create = <bpy_func Object.animation_data_create()>
obj.animation_visualization = bpy.data.objects['piece_0']...AnimViz
obj.asset_clear = <bpy_func Object.asset_clear()>
obj.asset_data = None
obj.asset_generate_preview = <bpy_func Object.asset_generate_preview()>
obj.asset_mark = <bpy_func Object.asset_mark()>
obj.bl_rna = <bpy_struct, Struct("Object") at 0x7f535b395a20>
obj.bound_box = bpy.data.objects['piece_0'].bound_box
obj.cache_release = <bpy_func Object.cache_release()>
obj.calc_matrix_camera = <bpy_func Object.calc_matrix_camera()>
obj.camera_fit_coords = <bpy_func Object.camera_fit_coords()>
obj.children = ()
obj.child

In [42]:




renderer = MyRenderer(cfg)


#save_dir = data_root_dir+'/results' #cfg.renderer.output_path
    
sampled_files = renderer.sample_data_files()
sampled_files = ['0']
# sampled_files = ["1"]

for file in sampled_files:
    print(file)
    transformation, gt_transformation, acc, init_pose, init_pose_centroid = renderer.load_transformation_data(file)
    print('==============transformation===============')
    print(transformation.shape)
    print('==============gt_transformation===============')
    print(gt_transformation.shape)
    print('==============init_pose===============')
    print(init_pose)
    parts = renderer.load_mesh_parts(file, gt_transformation, init_pose)
    print(file)
    print('==============parts===============')
    print(parts)
    save_path = cfg.renderer.output_path+f'{file}'
    os.makedirs(save_path, exist_ok=True)

    renderer.save_img(parts, gt_transformation, gt_transformation, init_pose, os.path.join(save_path, "gt.png"))

    if True:
        renderer.clean()
        break



    
    frame = 0

    # bpy.ops.wm.save_mainfile(filepath=save_path + "test" + '.blend')

    for i in range(transformation.shape[0]):
        renderer.render_parts(
            parts, 
            gt_transformation, 
            transformation[i], 
            init_pose, 
            frame,
        )
        frame += 1


    imgs_path = os.path.join(save_path, "imgs")
    os.makedirs(imgs_path, exist_ok=True)
    renderer.save_video(imgs_path=imgs_path, video_path=os.path.join(save_path, "video.mp4"), frame=frame)
    renderer.clean()


# In[ ]:



cycles rendering with: GPU
0
==============transformation===============
(20, 5, 10)
==============gt_transformation===============
(5, 10)
==============init_pose===============
[0.59839522 0.72763605 0.28771427 0.54838432 0.         0.83622643
 0.        ]
OBJ import of 'piece_0.obj' took 3.4 ms
OBJ import of 'piece_1.obj' took 5.1 ms
OBJ import of 'piece_2.obj' took 4.9 ms
OBJ import of 'piece_3.obj' took 1.8 ms
0
==============parts===============
[bpy.data.objects['piece_0'], bpy.data.objects['piece_1'], bpy.data.objects['piece_2'], bpy.data.objects['piece_3']]


TypeError: MyRenderer.save_img() missing 1 required positional argument: 'save_path'

In [ ]:








file = '0'

transformation, gt_transformation, acc, init_pose = renderer.load_transformation_data(file)
parts = renderer.load_mesh_parts(file, gt_transformation, init_pose)


if False:
    print(init_pose.shape)
    print(init_pose)
    print(transformation.shape)
    print(transformation)
    print(parts[0])

    #save_path = f"/work/users/j/a/jahn25/breaking-bad-dataset/results_render/{file}"
    #os.makedirs(save_path, exist_ok=True)
    #renderer.save_img(parts, gt_transformation, gt_transformation, init_pose, os.path.join(save_path, "gt.png"))

    #if True:
    #    renderer.clean()
    #    quit()

save_path = f"/work/users/j/a/jahn25/bio-dataset/results_render/{file}"
os.makedirs(save_path, exist_ok=True)
renderer.save_img(parts, gt_transformation, gt_transformation, init_pose, os.path.join(save_path, "gt.png"))

frame = 0

# bpy.ops.wm.save_mainfile(filepath=save_path + "test" + '.blend')

for i in range(transformation.shape[0]):
    renderer.render_parts(
        parts, 
        gt_transformation, 
        transformation[i], 
        init_pose, 
        frame,
    )
    frame += 1


imgs_path = os.path.join(save_path, "imgs")
os.makedirs(imgs_path, exist_ok=True)
renderer.save_video(imgs_path=imgs_path, video_path=os.path.join(save_path, "video.mp4"), frame=frame)
renderer.clean()



if True:
    quit()
# In[77]:


def debug_my():
    import numpy as np
    #everyday/BeerBottle/6da7fa9722b2a12d195232a03d04563a/fractured_1
    data_dict = np.load(os.path.join(project_home_dir, 'data/pc_data/everyday/val/00001.npz'))
    pc = data_dict['part_pcs_gt']
    data_id = data_dict['data_id'].item()
    part_valids = data_dict['part_valids']
    num_parts = data_dict["num_parts"].item()
    mesh_file_path = data_dict['mesh_file_path'].item()
    category = data_dict["category"]
    print('category',category)

    print('num_parts',num_parts)
    print('part_valids',part_valids)

    print('mesh_file_path',mesh_file_path)

    print('part_pcs_gt')
    print(pc.shape)
    print(pc)

debug_my(renderer.inference_path)


# In[5]:



# In[86]:


print(init_pose.shape)
print(init_pose)


# In[87]:


print(transformation.shape)
print(transformation)


# In[13]:


parts = renderer.load_mesh_parts(file, gt_transformation, init_pose)


# In[17]:


type(parts[0])


